# Prediction Analysis Notebook
This notebook covers **two** prediction tasks, each following the same pipeline:
1. Imports
2. Data Loading
3. Data Understanding (head / tail / info / describe)
4. Data Cleaning
5. Data Fill & Drop (handle missing values)
6. Graphs / Visualisation
7. Convert Data (encode categoricals, scale)
8. Train / Test Split
9. Train Model & Save to .pkl

**Task A:** House Price Prediction  (`house price  pridiction.csv`)
**Task B:** Spotify Song Popularity Prediction (`high_popularity_spotify_data.csv`)

> Run the cells in order. Every command lives in its own cell.

## 1. Imports
Import every library we need. Run this first so the rest of the notebook works.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import pickle

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
print('All libraries imported successfully.')

## 2. Data Loading
Load both CSV files into pandas DataFrames.

In [ ]:
house_df = pd.read_csv('house price  pridiction.csv')
spotify_df = pd.read_csv('high_popularity_spotify_data.csv')

print('House dataset shape :', house_df.shape)
print('Spotify dataset shape:', spotify_df.shape)

## 3. Data Understanding - House Price
Use `head()`, `tail()`, `info()`, `describe()` and `columns` to understand the data.

In [ ]:
# Show the first 5 rows
house_df.head()

Data Understanding - House Price (tail)
Show the last 5 rows with `tail()`.

In [ ]:
# Show the last 5 rows
house_df.tail()

Data Understanding - House Price (info)
Show column types and non-null counts with `info()`.

In [ ]:
# General information about the DataFrame
house_df.info()

Data Understanding - House Price (describe)
Statistical summary of numeric columns with `describe()`.

In [ ]:
# Statistical description
house_df.describe(include='all')

Data Understanding - House Price (columns & shape)
List all columns and the dataset shape.

In [ ]:
print('Columns:', list(house_df.columns))
print('Shape  :', house_df.shape)
print('Nulls per column:')
print(house_df.isnull().sum())

## 3. Data Understanding - Spotify Popularity
Repeat the understanding steps for the Spotify dataset.

In [ ]:
# First 5 rows
spotify_df.head()

Data Understanding - Spotify (tail)

In [ ]:
# Last 5 rows
spotify_df.tail()

Data Understanding - Spotify (info)

In [ ]:
spotify_df.info()

Data Understanding - Spotify (describe)

In [ ]:
spotify_df.describe(include='all')

Data Understanding - Spotify (columns & nulls)

In [ ]:
print('Columns:', list(spotify_df.columns))
print('Shape  :', spotify_df.shape)
print('Nulls per column:')
print(spotify_df.isnull().sum())

## 4. Data Cleaning - House Price
Extract numeric values from messy text columns (`price`, `rate`, `area`, `bedRoom`, etc.).

In [ ]:
# --- Helper to convert Indian price strings like '5.25 Crore' / '3.6 Crore' to float (in Crore) ---
def parse_price(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower().replace(',', '')
    m = re.search(r'\d+(?:\.\d+)?', val)
    if not m:
        return np.nan
    num = float(m.group())
    if 'crore' in val:
        return num
    if 'lakh' in val or 'lac' in val:
        return num / 100.0
    return num

# --- Helper to extract the FIRST number from a text column (handles '20,115/sq.ft.') ---
def first_number(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'\d+(?:\.\d+)?', str(val).replace(',', ''))
    return float(m.group()) if m else np.nan

house = house_df.copy()
house['price_crore'] = house['price'].apply(parse_price)
house['rate_per_sqft'] = house['rate'].apply(lambda x: first_number(x))
house['area_sqm'] = house['area'].apply(lambda x: first_number(x))
house['bedRoom_n'] = house['bedRoom'].apply(lambda x: first_number(x))
house['bathroom_n'] = house['bathroom'].apply(lambda x: first_number(x))
house['balcony_n'] = house['balcony'].apply(lambda x: first_number(x))
house['noOfFloor_n'] = house['noOfFloor'].apply(lambda x: first_number(x))
print('Cleaned numeric columns created:')
print(house[['price_crore','rate_per_sqft','area_sqm','bedRoom_n','bathroom_n','balcony_n','noOfFloor_n']].head())

Data Cleaning - House Price (drop useless text columns)
Remove columns that are free text / identifiers and not useful for modelling.

In [ ]:
text_cols_to_drop = ['property_name','link','society','price','rate','area','areaWithType',
                      'bedRoom','bathroom','balcony','additionalRoom','address','noOfFloor',
                      'facing','agePossession','nearbyLocations','description','furnishDetails',
                      'features','rating','property_id']
house = house.drop(columns=text_cols_to_drop)
print('Remaining house columns:', list(house.columns))

## 4. Data Cleaning - Spotify Popularity
Drop URL/ID/text columns that are not predictive and keep audio features + target.

In [ ]:
spotify = spotify_df.copy()
# Drop non-predictive text / identifier columns
drop_cols = ['track_href','uri','analysis_url','track_id','track_album_id','id',
             'playlist_id','track_name','track_artist','track_album_name','playlist_name',
             'track_album_release_date','type']
spotify = spotify.drop(columns=[c for c in drop_cols if c in spotify.columns])
print('Remaining spotify columns:', list(spotify.columns))

## 5. Data Fill & Drop - House Price
Drop rows where the target (`price_crore`) is missing, then fill remaining numeric nulls with the median.

In [ ]:
# Drop rows with no target value
house = house.dropna(subset=['price_crore'])
print('After dropping missing targets:', house.shape)

# Fill remaining numeric missing values with the column median
num_cols = house.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if house[col].isnull().any():
        house[col] = house[col].fillna(house[col].median())

print('Nulls remaining:')
print(house.isnull().sum())

Data Fill & Drop - Spotify Popularity
Drop rows missing the target, then fill numeric nulls with median and categorical nulls with mode.

In [ ]:
# Drop rows with no target
spotify = spotify.dropna(subset=['track_popularity'])
print('After dropping missing targets:', spotify.shape)

# Fill numeric nulls with median
num_cols = spotify.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if spotify[col].isnull().any():
        spotify[col] = spotify[col].fillna(spotify[col].median())

# Fill categorical nulls with mode
cat_cols = spotify.select_dtypes(include=['object','str']).columns
for col in cat_cols:
    if spotify[col].isnull().any():
        spotify[col] = spotify[col].fillna(spotify[col].mode()[0])

print('Nulls remaining:')
print(spotify.isnull().sum())

## 6. Graphs - House Price
Visualise the target distribution and relationships with numeric features.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(house['price_crore'], bins=30, kde=True, ax=axes[0,0])
axes[0,0].set_title('House Price Distribution (Crore)')

sns.scatterplot(data=house, x='area_sqm', y='price_crore', ax=axes[0,1])
axes[0,1].set_title('Area vs Price')

sns.boxplot(data=house, x='bedRoom_n', y='price_crore', ax=axes[1,0])
axes[1,0].set_title('Bedrooms vs Price')

sns.heatmap(house.corr(numeric_only=True), annot=True, fmt='.2f', ax=axes[1,1], cmap='coolwarm')
axes[1,1].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

Graphs - House Price (correlation with target)

In [ ]:
plt.figure(figsize=(8, 5))
corr = house.corr(numeric_only=True)['price_crore'].sort_values(ascending=False)
sns.barplot(x=corr.values, y=corr.index, palette='viridis')
plt.title('Feature correlation with House Price')
plt.tight_layout()
plt.show()

## 6. Graphs - Spotify Popularity
Visualise the popularity distribution and correlations of audio features.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(spotify['track_popularity'], bins=30, kde=True, ax=axes[0,0])
axes[0,0].set_title('Track Popularity Distribution')

sns.scatterplot(data=spotify, x='danceability', y='track_popularity', ax=axes[0,1])
axes[0,1].set_title('Danceability vs Popularity')

sns.boxplot(data=spotify, x='playlist_genre', y='track_popularity', ax=axes[1,0])
axes[1,0].set_title('Genre vs Popularity')

sns.heatmap(spotify.corr(numeric_only=True), annot=False, ax=axes[1,1], cmap='coolwarm')
axes[1,1].set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

Graphs - Spotify (popularity by genre)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=spotify, x='playlist_genre', y='track_popularity', estimator=np.mean)
plt.title('Average Popularity by Genre')
plt.tight_layout()
plt.show()

## 7. Convert Data - House Price
Separate features/target, encode any remaining categoricals, and scale numeric features.

In [ ]:
# Define target and features
y_house = house['price_crore']
X_house = house.drop(columns=['price_crore'])

# Encode any remaining object columns
house_encoders = {}
for col in X_house.select_dtypes(include=['object','str']).columns:
    le = LabelEncoder()
    X_house[col] = le.fit_transform(X_house[col].astype(str))
    house_encoders[col] = le

# Scale numeric features
house_scaler = StandardScaler()
X_house_scaled = pd.DataFrame(
    house_scaler.fit_transform(X_house),
    columns=X_house.columns
)
print('House features shape:', X_house_scaled.shape)
X_house_scaled.head()

## 7. Convert Data - Spotify Popularity
Encode categorical columns (`playlist_genre`, `playlist_subgenre`, `key`, `mode`) and scale features.

In [ ]:
# Define target and features
y_spot = spotify['track_popularity']
X_spot = spotify.drop(columns=['track_popularity'])

# Encode categorical columns
spot_encoders = {}
for col in X_spot.select_dtypes(include=['object','str']).columns:
    le = LabelEncoder()
    X_spot[col] = le.fit_transform(X_spot[col].astype(str))
    spot_encoders[col] = le

# Scale numeric features
spot_scaler = StandardScaler()
X_spot_scaled = pd.DataFrame(
    spot_scaler.fit_transform(X_spot),
    columns=X_spot.columns
)
print('Spotify features shape:', X_spot_scaled.shape)
X_spot_scaled.head()

## 8. Train / Test Split - House Price
Split the data into training (80%) and testing (20%) sets.

In [ ]:
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_house_scaled, y_house, test_size=0.2, random_state=42
)
print('Train shape:', Xh_train.shape, '| Test shape:', Xh_test.shape)

## 8. Train / Test Split - Spotify Popularity
Split the Spotify data into training and testing sets.

In [ ]:
Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_spot_scaled, y_spot, test_size=0.2, random_state=42
)
print('Train shape:', Xs_train.shape, '| Test shape:', Xs_test.shape)

## 9. Train Model & Save - House Price
Train a Random Forest regressor, evaluate it, and save the model + scaler to a `.pkl` file.

In [ ]:
house_model = RandomForestRegressor(n_estimators=100, random_state=42)
house_model.fit(Xh_train, yh_train)

h_pred = house_model.predict(Xh_test)
print('House R2  :', round(r2_score(yh_test, h_pred), 4))
print('House MAE :', round(mean_absolute_error(yh_test, h_pred), 4))
print('House RMSE:', round(np.sqrt(mean_squared_error(yh_test, h_pred)), 4))

# Save model, scaler and feature columns
with open('house_price_model.pkl', 'wb') as f:
    pickle.dump({'model': house_model,
                  'scaler': house_scaler,
                  'encoders': house_encoders,
                  'features': list(X_house.columns)}, f)
print('House model saved to house_price_model.pkl')

## 9. Train Model & Save - Spotify Popularity
Train a Random Forest regressor for song popularity, evaluate it, and save to `.pkl`.

In [ ]:
spotify_model = RandomForestRegressor(n_estimators=100, random_state=42)
spotify_model.fit(Xs_train, ys_train)

s_pred = spotify_model.predict(Xs_test)
print('Spotify R2  :', round(r2_score(ys_test, s_pred), 4))
print('Spotify MAE :', round(mean_absolute_error(ys_test, s_pred), 4))
print('Spotify RMSE:', round(np.sqrt(mean_squared_error(ys_test, s_pred)), 4))

# Save model, scaler and feature columns
with open('spotify_popularity_model.pkl', 'wb') as f:
    pickle.dump({'model': spotify_model,
                  'scaler': spot_scaler,
                  'encoders': spot_encoders,
                  'features': list(X_spot.columns)}, f)
print('Spotify model saved to spotify_popularity_model.pkl')

## 10. (Bonus) Load & Use Saved Model
Example of loading a saved `.pkl` model and making a prediction on new data.

In [ ]:
# Load the saved house price model
with open('house_price_model.pkl', 'rb') as f:
    saved = pickle.load(f)

# Build a small sample using the SAME feature order/columns
sample = pd.DataFrame([{c: 0 for c in saved['features']}])
sample_scaled = saved['scaler'].transform(sample)
prediction = saved['model'].predict(sample_scaled)
print('Sample prediction (Crore):', prediction)